# Modelo final: KNN con spline de mora

Este notebook reproduce el modelo elegido para compartirlo y generar un submission de Kaggle.

**Modelo:** KNN con 61 vecinos, ponderación por distancia y spline cúbico de 3 nudos sobre mora.

**Evidencia disponible:** AUC de validación 0,8215 y AUC público de Kaggle 0,8199. La selección se realizó en notebooks anteriores; aquí no se exploran hiperparámetros para mantener el flujo sencillo y reproducible.

## 1. Importar librerías y ubicar el proyecto

La primera parte solo importa herramientas y define rutas. El código funciona tanto si se ejecuta desde la carpeta raíz como desde `notebooks/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

RAIZ = Path.cwd().resolve()
if RAIZ.name == 'notebooks':
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import transformaciones as tr

RANDOM_STATE = 2026
K_FINAL = 61
CONFIGURACION_FINAL = 'spline_mora_3_nudos'
RUTA_DATOS = RAIZ / 'data' / 'processed'
RUTA_SALIDA = RAIZ / 'outputs' / 'predicciones' / 'modelo_final'
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

## 2. Cargar train y test

`train_preparado.csv` contiene la variable objetivo `resultado`. `test_preparado.csv` tiene las mismas variables, pero sin esa columna; por eso es la base que recibirá las predicciones finales.

In [ ]:
train_original = pd.read_csv(RUTA_DATOS / 'train_preparado.csv')
test_original = pd.read_csv(RUTA_DATOS / 'test_preparado.csv')

display(pd.DataFrame({
    'base': ['train', 'test'],
    'clientes': [len(train_original), len(test_original)],
    'incluye resultado': ['sí', 'no']
}))

## 3. Aplicar la transformación ganadora

La configuración `spline_mora_3_nudos` conserva las variables originales y reemplaza la relación lineal de mora por una representación suave y no lineal. Los tres nudos permiten que el riesgo cambie de forma gradual en distintas zonas de mora.

El preprocesador aprende imputación, escalamiento, codificación y spline solamente con los datos de entrenamiento.

In [ ]:
configuracion = tr.obtener_configuracion(CONFIGURACION_FINAL)
train = tr.aplicar_transformaciones(train_original, configuracion)
test = tr.aplicar_transformaciones(test_original, configuracion)

numericas, splines, categoricas = tr.obtener_grupos(configuracion)
PREDICTORES = numericas + splines + categoricas

X = train[PREDICTORES]
y = train['resultado']
X_test = test[PREDICTORES]

display(pd.DataFrame({
    'tipo': ['numéricas lineales', 'variables spline', 'categóricas'],
    'variables': [', '.join(numericas), ', '.join(splines), ', '.join(categoricas)]
}))

## 4. Validar el modelo con clientes no vistos

Esta partición reproduce el esquema usado durante la selección: 80% para entrenamiento y 20% para validación, conservando la proporción de churn. El resultado es una comprobación local; la métrica definitiva del concurso sigue siendo el AUC calculado por Kaggle.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

modelo_knn = Pipeline([
    ('preprocesar', tr.crear_preprocesador(configuracion)),
    ('knn', KNeighborsClassifier(n_neighbors=K_FINAL, weights='distance'))
])

modelo_knn.fit(X_train, y_train)
prob_val = modelo_knn.predict_proba(X_val)[:, 1]
auc_validacion = roc_auc_score(y_val, prob_val)

display(pd.DataFrame({
    'modelo': ['KNN final'],
    'configuración': [CONFIGURACION_FINAL],
    'k': [K_FINAL],
    'pesos': ['distance'],
    'AUC validación': [auc_validacion]
}).style.format({'AUC validación': '{:.4f}'}))

## 5. Entrenar con todos los clientes y crear el submission

Después de validar, entrenamos el mismo pipeline con todo `train`. La salida contiene exactamente las dos columnas requeridas por Kaggle: `id` y `resultado`. `resultado` es una probabilidad de churn, no una etiqueta 0/1.

In [ ]:
modelo_final = Pipeline([
    ('preprocesar', tr.crear_preprocesador(configuracion)),
    ('knn', KNeighborsClassifier(n_neighbors=K_FINAL, weights='distance'))
])
modelo_final.fit(X, y)

submission = pd.DataFrame({
    'id': test['id'],
    'resultado': modelo_final.predict_proba(X_test)[:, 1]
})

assert list(submission.columns) == ['id', 'resultado']
assert len(submission) == len(test)
assert submission.resultado.between(0, 1).all()

ARCHIVO_SUBMISSION = RUTA_SALIDA / 'submission_knn_spline_mora_3_nudos_k61.csv'
submission.to_csv(ARCHIVO_SUBMISSION, index=False)
display(submission.head())
print(f'Submission guardado en: {ARCHIVO_SUBMISSION}')

## Resumen

El modelo final usa mora de forma no lineal mediante un spline y combina la evidencia de los 61 clientes históricos más cercanos, con mayor peso para quienes están más cerca. El archivo generado se puede subir directamente a Kaggle.